In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

### L'objectif de ce pre-processing est de faire le plus classiquement possible

In [3]:
# les valeurs nuls
df = pd.read_csv('../data/kaggle_b2_fraud_train.csv')
data_test =  pd.read_csv('../data/kaggle_b2_fraud_test.csv')


In [4]:
df['target_is_fraud']

0         0
1         0
2         0
3         0
4         0
         ..
159995    0
159996    0
159997    0
159998    0
159999    0
Name: target_is_fraud, Length: 160000, dtype: int64

In [5]:
df.head()

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,mostly_missing_col,manual_review_result,post_event_status_code,occupation,device_type,merchant_category,signup_date,secondary_email,chargeback_resolution_time_days,legacy_partner_score
0,CUST_7O7B4OC888,ACC_TACKQKWHFB2I,18,42,12604.31,715.0,18,41.22,253.57,14.2,...,NaN,approve,0,freelancer,phone,digital_services,2025-01-06,NaN,11.3,NaN
1,CUST_G5D9HCKEYD,ACC_EN7OC1DLMQUQ,25,11,36133.60,675.0,19,65.60,523.45,1.5,...,NaN,approve,0,employee,phone,gaming,2025-08-04,NaN,3.0,NaN
2,CUST_5YHBCQWI3O,ACC_J38B0G2XTCUS,23,11,29528.32,667.0,28,NaN,258.13,31.5,...,NaN,approve,0,retired,desktop,groceries,2024-07-25,NaN,2.1,NaN
3,CUST_RF6DJEUOOU,ACC_6F99QLXXUP0X,18,6,28247.85,630.0,18,198.03,316.96,5.0,...,NaN,review,0,public_sector,desktop,sports,2025-11-26,NaN,1.2,NaN
4,CUST_WNVVULGQE7,ACC_91BZO39DAE2T,30,48,13978.11,626.0,21,127.85,851.10,37.2,...,NaN,approve,0,employee,phone,electronics,2024-06-09,NaN,4.4,NaN


## suppressio des variables catégorielles 

In [6]:
# 1. SUPPRIMER TOUTES LES COLONNES CATÉGORIELLES D'ABORD
df = df.select_dtypes(include=['number'])

print(f"Shape après suppression catégorielles : {df.shape}")
print(f"Colonnes restantes : {list(df.columns)}")

Shape après suppression catégorielles : (160000, 36)
Colonnes restantes : ['age', 'tenure_months', 'annual_income_eur', 'credit_score', 'num_transactions_30d', 'avg_amount_30d_eur', 'max_amount_30d_eur', 'days_since_last_login', 'support_tickets_90d', 'chargebacks_12m', 'failed_payments_6m', 'device_trust_z', 'ip_risk_z', 'is_vpn', 'num_devices_30d', 'is_new_device', 'postal_code', 'target_is_fraud', 'annual_income_log', 'income_dup_eur', 'credit_score_scaled', 'tx_amount_total_30d_eur', 'max_to_avg_ratio', 'noise_1', 'noise_2', 'noise_3', 'noise_4', 'noise_5', 'noise_6', 'noise_7', 'noise_8', 'constant_flag', 'mostly_missing_col', 'post_event_status_code', 'chargeback_resolution_time_days', 'legacy_partner_score']


## suppretion des colonnes avec trop de données manquantes

In [7]:
nan_counts = df.isna().sum().sort_values(ascending=False)

nan_percentage = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
nan_percentage.head()

mostly_missing_col      96.950625
legacy_partner_score    95.945000
annual_income_eur        7.045000
avg_amount_30d_eur       5.946875
credit_score             5.051250
dtype: float64

In [8]:
df = df.drop(columns=({"mostly_missing_col", "legacy_partner_score" }))

In [9]:
nan_percentage = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
nan_percentage.head(10)

annual_income_eur        7.045000
avg_amount_30d_eur       5.946875
credit_score             5.051250
max_amount_30d_eur       5.011875
device_trust_z           3.918125
ip_risk_z                3.061875
tenure_months            0.000000
age                      0.000000
num_transactions_30d     0.000000
days_since_last_login    0.000000
dtype: float64

## imputation par la médiane des données manquantes restantes

In [10]:
# Imputation directe par la médiane
cols_to_impute = [
    'annual_income_eur',
    'avg_amount_30d_eur', 
    'credit_score',
    'max_amount_30d_eur',
    'device_trust_z',
    'ip_risk_z'
]

# CORRECT : juste modifier les colonnes, pas écraser df
df[cols_to_impute] = df[cols_to_impute].fillna(df[cols_to_impute].median())

print("Vérification - NaN restants :")
print(df[cols_to_impute].isna().sum())

Vérification - NaN restants :
annual_income_eur     0
avg_amount_30d_eur    0
credit_score          0
max_amount_30d_eur    0
device_trust_z        0
ip_risk_z             0
dtype: int64


In [11]:
df['target_is_fraud']

0         0
1         0
2         0
3         0
4         0
         ..
159995    0
159996    0
159997    0
159998    0
159999    0
Name: target_is_fraud, Length: 160000, dtype: int64

## supprimer les doublons

In [12]:
# Vérifier et supprimer les doublons
print(f"Doublons avant : {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape après suppression doublons : {df.shape}")

Doublons avant : 0
Shape après suppression doublons : (160000, 34)


## scaler simplement avec un standarscaler

In [13]:
# Scaler tout sauf la target
cols_to_scale = [col for col in df.columns if col != 'target_is_fraud']

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
df.head()

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,noise_2,noise_3,noise_4,noise_5,noise_6,noise_7,noise_8,constant_flag,post_event_status_code,chargeback_resolution_time_days
0,-1.729818,1.349449,-0.877591,0.772230,-0.134734,-0.489097,-0.167778,0.184663,0.223405,-0.224442,...,-0.833142,0.964596,1.451607,0.113477,0.612024,-0.268804,0.241549,0.0,-0.296831,0.806031
1,-1.132460,-0.388951,-0.043677,0.087114,-0.107325,0.115393,0.912675,-0.878831,-0.893551,4.243175,...,-0.925067,0.336684,0.094246,1.650957,1.141504,0.600553,-0.198639,0.0,-0.296831,-0.678103
2,-1.303134,-0.388951,-0.277778,-0.049909,0.139358,-0.210654,-0.149523,1.633360,0.223405,-0.224442,...,-1.889742,-0.360250,-2.010010,-0.289967,-0.238822,1.443554,-3.172710,0.0,-0.296831,-0.839033
3,-1.729818,-0.669338,-0.323160,-0.683641,-0.134734,3.398929,0.086001,-0.585742,0.223405,-0.224442,...,1.111545,-0.437378,-0.606572,-0.066508,1.810603,1.379443,-0.097172,0.0,-0.296831,-0.999963
4,-0.705776,1.685913,-0.828902,-0.752153,-0.052506,1.658851,2.224408,2.110676,-0.893551,-0.224442,...,0.599123,-1.342041,0.165596,0.589740,0.493524,-1.434491,0.697380,0.0,-0.296831,-0.427767


In [ ]:
df.to_csv("1_paul_preprocess.csv", sep=",", index=False)